# AeroPure — Week 8: Champion XGBoost Models & SHAP Explainability Engine
**Final Milestone of the AeroPure ML Roadmap**

### 1. Overview
Week 8 implements the champion gradient boosted tree models (`XGBRegressor` and `XGBClassifier`) with early stopping, performs the complete multi-model benchmark across Weeks 3–8, and explains predictions using Shapley Additive exPlanations (SHAP).


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

sys.path.insert(0, os.path.abspath(".."))
from src.regression import train_xgboost_regressor
from src.classification import train_xgboost_classifier
from src.explainability import compute_shap_explanations, explain_individual_prediction, generate_natural_language_explanation

PROC_PATH = os.path.join("..", "data", "processed_data.csv")
if os.path.exists(PROC_PATH):
    from src.feature_engineering import prepare_time_series_splits
    df_proc = pd.read_csv(PROC_PATH)
    (X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf, _, _) = prepare_time_series_splits(df_proc)
    print("Data loaded successfully.")
else:
    print("Processed dataset not found.")


### 2. Training Champion XGBoost Regressor & Classifier


In [ ]:
if "X_train" in locals():
    xgb_reg, xgb_reg_metrics, xgb_reg_preds, xgb_reg_imp = train_xgboost_regressor(
        X_train, y_train_reg, X_test, y_test_reg, n_estimators=250, learning_rate=0.05, max_depth=5
    )
    print("XGBoost Regressor Metrics:", xgb_reg_metrics)
    
    xgb_clf, xgb_clf_metrics, xgb_clf_preds, xgb_clf_probs, xgb_clf_cm, xgb_clf_imp = train_xgboost_classifier(
        X_train, y_train_clf, X_test, y_test_clf, n_estimators=250, learning_rate=0.05, max_depth=5
    )
    print("XGBoost Classifier Metrics:", xgb_clf_metrics)


### 3. Comprehensive Model Benchmark Across Weeks 3–8
Comparing all regression models (OLS, Ridge, Lasso, Decision Tree, Random Forest, XGBoost) and all classification models.


In [ ]:
METRICS_PATH = os.path.join("..", "outputs", "metrics", "regression_leaderboard.csv")
if os.path.exists(METRICS_PATH):
    reg_leaderboard = pd.read_csv(METRICS_PATH)
    print("REGRESSION LEADERBOARD:")
    display(reg_leaderboard)
    
    clf_leaderboard = pd.read_csv(os.path.join("..", "outputs", "metrics", "classification_leaderboard.csv"))
    print("\nCLASSIFICATION LEADERBOARD:")
    display(clf_leaderboard)


### 4. SHAP Explainability Engine
Using `shap.TreeExplainer` on the XGBoost champion model to compute exact Shapley attributions for global and local predictions.


In [ ]:
if "xgb_reg" in locals():
    explainer = shap.TreeExplainer(xgb_reg)
    shap_values = explainer(X_test)
    
    plt.figure(figsize=(10, 6), dpi=120)
    shap.plots.beeswarm(shap_values, max_display=12, show=False)
    plt.title("SHAP Beeswarm Summary — Feature Impact on Next-Day AQI")
    plt.tight_layout()
    plt.show()


### 5. Local Prediction Waterfall Explanation (High-Risk Day vs Clean Day)


In [ ]:
if "shap_values" in locals():
    max_idx = int(np.argmax(xgb_reg_preds))
    plt.figure(figsize=(10, 5), dpi=120)
    shap.plots.waterfall(shap_values[max_idx], max_display=10, show=False)
    plt.title("SHAP Waterfall: Hazardous Air Prediction Drivers")
    plt.tight_layout()
    plt.show()


### Week 8 Final Conclusion
- XGBoost achieved champion status across both regression (lowest RMSE, highest $R^2$) and classification (highest F1 and ROC-AUC).
- SHAP provided mathematically principled, transparent explanations translating ML tree splits into actionable civic advisories.
- The AeroPure Week 1–8 roadmap is complete and ready for presentation.
